# ComfyUI + comfyweb — Z-Image Turbo (Colab)

Corré **ComfyUI** con GPU gratis en Google Colab y usalo desde **comfyweb** (una UI web simple).
Este notebook levanta ComfyUI con Z-Image Turbo + ControlNet + upscale, y te da una **URL pública**
para conectar tu comfyweb.

**Cómo usarlo:**
1. Arriba: *Entorno de ejecución → Cambiar tipo de entorno →* elegí la GPU:
   - **T4** (gratis) alcanza para **Crear, ControlNet, LoRA, Describir imagen y Restaurar**.
   - **L4** solo hace falta para el modo **Editar** (Qwen-Image-Edit): en T4 se queda sin memoria (OOM).
2. *Entorno de ejecución → Ejecutar todo.* La 1ª vez baja los modelos a **tu** Google Drive
   (~14 GB, tarda); las siguientes veces arranca en ~2-3 min.
3. Al final imprime una **URL pública** `https://...trycloudflare.com`.
   Pegála en comfyweb → botón *Colab (Cloudflare)* → *Guardar y Conectar*.

> La URL cambia cada vez que reiniciás el notebook — es normal, la volvés a pegar.

## 1) Montar Google Drive (los modelos viven acá, no se rebajan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Carpeta en tu Drive (se crea sola la 1ª vez). Los modelos viven acá y no se rebajan.
DRIVE_BASE = '/content/drive/MyDrive/ComfyUI-Colab'
MODELS = os.path.join(DRIVE_BASE, 'models')
for sub in ['diffusion_models','text_encoders','vae','model_patches','upscale_models','loras','LLM']:
    os.makedirs(os.path.join(MODELS, sub), exist_ok=True)
print('Drive montado. Modelos en:', MODELS)

## 2) Clonar ComfyUI + custom nodes (Z-Image + ControlNet + upscale)

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# Custom nodes que usa el pipeline DIPLO (ControlNet preprocesadores, tiles, etc.)
CN = '/content/ComfyUI/custom_nodes'
repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager',
    'https://github.com/Fannovel16/comfyui_controlnet_aux',   # AIO_Preprocessor
    'https://github.com/TTPlanetPig/Comfyui_TTP_Toolset',     # tiles
    'https://github.com/yolain/ComfyUI-Easy-Use',
    'https://github.com/rgthree/rgthree-comfy',
    'https://github.com/LAOGOU-666/Comfyui-Memory_Cleanup',   # RAMCleanup
    'https://github.com/city96/ComfyUI-GGUF',  # UnetLoaderGGUF (carga el Qwen-Image-Edit GGUF)
    'https://github.com/KLL535/ComfyUI_Simple_Qwen3-VL-gguf',  # SimpleQwenVLgguf: imagen -> prompt (Qwen3-VL)
]
for u in repos:
    name = u.rstrip('/').split('/')[-1]
    p = os.path.join(CN, name)
    if not os.path.isdir(p):
        !git clone --depth 1 {u} {p}
    req = os.path.join(p, 'requirements.txt')
    if os.path.isfile(req):
        !pip install -q -r {req}
print('\nComfyUI + custom nodes listos')

# --- Mini endpoint para agregar LoRAs por URL desde comfyweb (POST /leo/add_lora) ---
import os as _os
_os.makedirs('/content/ComfyUI/custom_nodes/leo_lora_fetch', exist_ok=True)
with open('/content/ComfyUI/custom_nodes/leo_lora_fetch/__init__.py','w') as _f:
    _f.write(r"""
import os, urllib.request
from aiohttp import web
import server, folder_paths

@server.PromptServer.instance.routes.post("/leo/add_lora")
async def leo_add_lora(request):
    try:
        data = await request.json()
    except Exception:
        data = {}
    url = (data.get("url") or "").strip()
    filename = (data.get("filename") or "").strip()
    if not url:
        return web.json_response({"success": False, "error": "falta url"}, status=400)
    dirs = folder_paths.get_folder_paths("loras")
    target_dir = dirs[0] if dirs else "/content/ComfyUI/models/loras"
    os.makedirs(target_dir, exist_ok=True)
    if not filename:
        filename = url.split("?")[0].rstrip("/").split("/")[-1] or "lora.safetensors"
    if not filename.lower().endswith((".safetensors", ".pt", ".ckpt", ".bin")):
        filename += ".safetensors"
    dest = os.path.join(target_dir, filename)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as out:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)
    except Exception as e:
        return web.json_response({"success": False, "error": str(e)}, status=500)
    try:
        if hasattr(folder_paths, "cache_helper"):
            folder_paths.cache_helper.cache.clear()
    except Exception:
        pass
    try:
        if hasattr(folder_paths, "filename_list_cache"):
            folder_paths.filename_list_cache.clear()
    except Exception:
        pass
    try:
        folder_paths.get_filename_list("loras")
    except Exception:
        pass
    return web.json_response({"success": True, "name": filename, "path": dest})

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
""")
print('leo_lora_fetch instalado (endpoint /leo/add_lora)')


# --- Qwen3-VL (SimpleQwenVLgguf): llama-cpp-python con CUDA + soporte Qwen3-VL ---
# El pip normal NO trae Qwen3-VL. Wheel pre-compilado de JamePeng (cu128/cp312/linux), resuelto
# por la API de releases. Si falla, ComfyUI igual arranca (Z-Image OK), solo sin el nodo Qwen-VL.
import urllib.request, json as _json
def _install_llamacpp():
    try:
        rels=_json.load(urllib.request.urlopen('https://api.github.com/repos/JamePeng/llama-cpp-python/releases?per_page=25', timeout=30))
    except Exception as e:
        print('No pude leer releases JamePeng:', e); return
    cands=[]
    for r in rels:
        if 'cu128' in r['tag_name'] and 'linux' in r['tag_name']:
            for a in r.get('assets', []):
                n=a['name']
                if n.endswith('.whl') and 'cp312' in n and 'linux' in n:
                    cands.append((('AVX2' in n.upper()), a['browser_download_url']))
    if not cands:
        print('Sin wheel cu128/cp312/linux; pip normal (puede no soportar Qwen3-VL)')
        get_ipython().system('pip install -q llama-cpp-python'); return
    cands.sort(reverse=True)
    url=cands[0][1]; print('llama-cpp-python wheel:', url.split('/')[-1])
    get_ipython().system(f'pip install -q "{url}"')
_install_llamacpp()
get_ipython().system('pip install -q json_repair colorama pillow')
print('Qwen-VL (SimpleQwenVLgguf) deps listas')


## 3) Descargar modelos a Drive (solo la 1ª vez)

Si ya están en tu Drive, los saltea. Fuentes oficiales (Apache-2.0).

In [ ]:
import os

def dl(url, dest, min_mb=1):
    if os.path.isfile(dest) and os.path.getsize(dest) > min_mb*1_000_000:
        print('✓ ya está:', os.path.basename(dest)); return
    print('↓ bajando:', os.path.basename(dest))
    !wget -q --show-progress -O "{dest}" "{url}"

MODELS = '/content/drive/MyDrive/ComfyUI-Colab/models'

# Difusión (Z-Image Turbo fp8 e4m3fn, ideal para T4)
dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors',
   f'{MODELS}/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors', 500)
# Text encoder (Qwen3-4B fp8) — CLIPLoader tipo lumina2
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors',
   f'{MODELS}/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 500)
# VAE
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors',
   f'{MODELS}/vae/ae.safetensors', 50)
# ControlNet Union (Canny/Depth/Pose/etc.)
dl('https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union.safetensors',
   f'{MODELS}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors', 500)
# Upscaler 2x
dl('https://huggingface.co/Kim2091/2x-AnimeSharpV4/resolve/main/2x-AnimeSharpV4_RCAN.safetensors',
   f'{MODELS}/upscale_models/2x-AnimeSharpV4_RCAN.safetensors', 5)

print('\nModelos OK en Drive')

# Qwen3-VL-8B (vision -> prompt, nodo SimpleQwenVLgguf). A Drive/models/LLM (5.0 GB + 0.75 GB).
LLM = '/content/drive/MyDrive/ComfyUI-Colab/models/LLM'
os.makedirs(LLM, exist_ok=True)
dl('https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3VL-8B-Instruct-Q4_K_M.gguf',
   f'{LLM}/Qwen3VL-8B-Instruct-Q4_K_M.gguf', 500)
dl('https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf',
   f'{LLM}/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf', 100)
print('Qwen3-VL GGUF OK en Drive/models/LLM')


# Qwen-Image-Edit-2509 (EDITAR imagen: instrucción -> nueva imagen).
# GGUF Q4 (13GB) para entrar en Drive; nodos ya en el core de ComfyUI. Ideal con GPU L4.
dl('https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/resolve/main/Qwen-Image-Edit-2509-Q4_K_M.gguf',
   f'{MODELS}/diffusion_models/Qwen-Image-Edit-2509-Q4_K_M.gguf', 1000)
dl('https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors',
   f'{MODELS}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', 1000)
dl('https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors',
   f'{MODELS}/vae/qwen_image_vae.safetensors', 50)
dl('https://huggingface.co/lightx2v/Qwen-Image-Lightning/resolve/main/Qwen-Image-Lightning-4steps-V1.0.safetensors',
   f'{MODELS}/loras/Qwen-Image-Lightning-4steps-V1.0.safetensors', 100)
print('Qwen-Image-Edit OK en Drive')

## 4) Apuntar ComfyUI a los modelos de Drive + lanzar + túnel público

Al final imprime la **URL pública**. Pegála en comfyweb.

In [ ]:
# ComfyUI lee los modelos desde Drive (no copia, los usa en su lugar)
yaml = '''colab_drive:
  base_path: /content/drive/MyDrive/ComfyUI-Colab/models
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: text_encoders
  vae: vae
  model_patches: model_patches
  upscale_models: upscale_models
  loras: loras
'''
open('/content/ComfyUI/extra_model_paths.yaml','w').write(yaml)
print('extra_model_paths.yaml escrito')

# ============================================================
#  TUNEL PUBLICO (Cloudflare quick tunnel, gratis, sin cuenta).
#  Imprime una URL https://...trycloudflare.com -> pegala en comfyweb.
#  --protocol http2: en Colab el QUIC/UDP es inestable y el tunel se corta
#  en loop; forzar HTTP/2 (TCP) lo mantiene estable.
# ============================================================
import threading, time, socket, subprocess, os, re
%cd /content/ComfyUI

# Descargar cloudflared (una vez)
if not os.path.isfile('/usr/local/bin/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')

def wait_8188():
    while socket.socket().connect_ex(('127.0.0.1', 8188)) != 0:
        time.sleep(1)

def tunnel():
    wait_8188()
    proc = subprocess.Popen(
        ['cloudflared','tunnel','--no-autoupdate','--protocol','http2',
         '--url','http://127.0.0.1:8188'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    shown = False
    for line in proc.stdout:
        m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
        if m and not shown:
            shown = True
            print('\n\n' + '='*56)
            print('  URL PUBLICA (pegala en comfyweb, boton "Colab"):')
            print('  ' + m.group(0))
            print('='*56 + '\n')

threading.Thread(target=tunnel, daemon=True).start()

!python main.py --dont-print-server